# Phase 3: Sales Analysis

This notebook calculates the overall KPIs, time-based sales trends, and product performance required for Phase 3.

## 1. Import libraries and load the cleaned dataset

In [1]:
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()

if current_folder.name == 'notebooks':
    project_folder = current_folder.parent
else:
    project_folder = current_folder

cleaned_file = project_folder / 'data' / 'processed' / 'cleaned_sales.csv'
sales = pd.read_csv(cleaned_file, parse_dates=['Order Date', 'Delivery Date'])

print('Rows loaded:', len(sales))

Rows loaded: 62884


## 2. Calculate the overall KPIs

In [2]:
total_revenue = sales['Sales USD'].sum()
total_profit = sales['Profit USD'].sum()
total_orders = sales['Order Number'].nunique()
total_customers = sales['CustomerKey'].nunique()
average_order_value = total_revenue / total_orders
profit_margin = (total_profit / total_revenue) * 100

kpi_summary = pd.DataFrame({
    'KPI': [
        'Total Revenue',
        'Total Profit',
        'Total Orders',
        'Total Customers',
        'Average Order Value',
        'Profit Margin %',
    ],
    'Value': [
        round(total_revenue, 2),
        round(total_profit, 2),
        total_orders,
        total_customers,
        round(average_order_value, 2),
        round(profit_margin, 2),
    ],
})

display(kpi_summary)

,KPI,Value
0,Total Revenue,55755479.59
1,Total Profit,32662688.38
2,Total Orders,26326.00
3,Total Customers,11887.00
4,Average Order Value,2117.89
5,Profit Margin %,58.58


## 3. Analyze yearly sales

In [3]:
yearly_sales = (
    sales.groupby('Year', as_index=False)['Sales USD']
    .sum()
    .sort_values('Year')
)

yearly_sales['Growth %'] = yearly_sales['Sales USD'].pct_change() * 100
yearly_sales['Sales USD'] = yearly_sales['Sales USD'].round(2)
yearly_sales['Growth %'] = yearly_sales['Growth %'].round(2)

display(yearly_sales)

,Year,Sales USD,Growth %
0,2016,6946793.56,NaN
1,2017,7421422.27,6.83
2,2018,12788960.66,72.32
3,2019,18264382.48,42.81
4,2020,9294632.14,-49.11
5,2021,1039288.48,-88.82


In [4]:
complete_growth_rows = yearly_sales.dropna(subset=['Growth %'])
highest_growth_row = complete_growth_rows.loc[complete_growth_rows['Growth %'].idxmax()]

print('Year with maximum growth:', int(highest_growth_row['Year']))
print('Maximum growth percentage:', highest_growth_row['Growth %'])

Year with maximum growth: 2018
Maximum growth percentage: 72.32


## 4. Analyze quarterly sales

In [5]:
quarterly_sales = (
    sales.groupby(['Year', 'Quarter'], as_index=False)['Sales USD']
    .sum()
    .sort_values(['Year', 'Quarter'])
)

quarterly_sales['Sales USD'] = quarterly_sales['Sales USD'].round(2)
display(quarterly_sales)

,Year,Quarter,Sales USD
0,2016,1,1879424.44
1,2016,2,1225163.09
2,2016,3,1569893.64
3,2016,4,2272312.39
4,2017,1,1751889.58
5,2017,2,1306570.33
6,2017,3,1791540.87
7,2017,4,2571421.49
8,2018,1,2696369.20
9,2018,2,2117868.43


## 5. Analyze monthly sales

In [6]:
monthly_sales = (
    sales.groupby(['Year', 'Month', 'Month Name'], as_index=False)['Sales USD']
    .sum()
    .sort_values(['Year', 'Month'])
)

monthly_sales['Sales USD'] = monthly_sales['Sales USD'].round(2)
display(monthly_sales)

,Year,Month,Month Name,Sales USD
0,2016,1,January,649918.78
1,2016,2,February,891098.30
2,2016,3,March,338407.36
3,2016,4,April,110591.63
4,2016,5,May,595986.18
...,...,...,...,...
57,2020,10,October,245647.59
58,2020,11,November,256701.02
59,2020,12,December,651526.44
60,2021,1,January,513021.58


In [7]:
sales_by_month_name = (
    sales.groupby(['Month', 'Month Name'], as_index=False)['Sales USD']
    .sum()
    .sort_values('Month')
)

sales_by_month_name['Sales USD'] = sales_by_month_name['Sales USD'].round(2)
highest_month = sales_by_month_name.loc[sales_by_month_name['Sales USD'].idxmax()]
lowest_month = sales_by_month_name.loc[sales_by_month_name['Sales USD'].idxmin()]

display(sales_by_month_name)
print('Month with highest sales:', highest_month['Month Name'])
print('Month with lowest sales:', lowest_month['Month Name'])

,Month,Month Name,Sales USD
0,1,January,6759981.20
1,2,February,7842476.23
2,3,March,2625522.85
3,4,April,607334.05
4,5,May,4757983.80
5,6,June,4293036.54
6,7,July,3852415.81
7,8,August,4085169.32
8,9,September,4363863.61
9,10,October,4315027.44


Month with highest sales: February
Month with lowest sales: April


## 6. Check seasonality

Average monthly sales are used so January and February are not overstated by the additional partial year in 2021.

In [8]:
seasonality = (
    monthly_sales.groupby(['Month', 'Month Name'], as_index=False)['Sales USD']
    .mean()
    .rename(columns={'Sales USD': 'Average Monthly Sales'})
    .sort_values('Month')
)

seasonality['Average Monthly Sales'] = seasonality['Average Monthly Sales'].round(2)
highest_average_month = seasonality.loc[seasonality['Average Monthly Sales'].idxmax()]
lowest_average_month = seasonality.loc[seasonality['Average Monthly Sales'].idxmin()]

display(seasonality)
print('Highest average sales month:', highest_average_month['Month Name'])
print('Lowest average sales month:', lowest_average_month['Month Name'])
print('The difference between monthly averages shows a seasonal sales pattern.')

,Month,Month Name,Average Monthly Sales
0,1,January,1126663.53
1,2,February,1307079.37
2,3,March,525104.57
3,4,April,121466.81
4,5,May,951596.76
5,6,June,858607.31
6,7,July,770483.16
7,8,August,817033.86
8,9,September,872772.72
9,10,October,863005.49


Highest average sales month: December
Lowest average sales month: April
The difference between monthly averages shows a seasonal sales pattern.


## 7. Analyze sales by category

In [9]:
category_performance = (
    sales.groupby('Category', as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
    .sort_values('Revenue', ascending=False)
)

category_performance[['Revenue', 'Profit']] = category_performance[['Revenue', 'Profit']].round(2)
display(category_performance)

,Category,Revenue,Profit,Orders
3,Computers,19301595.46,11277447.90,10990
5,Home Appliances,10795478.59,6296338.85,5195
1,Cameras and camcorders,6520168.02,3919800.99,4995
2,Cell phones,6183791.22,3498626.54,8442
7,TV and Video,5928982.69,3536694.39,3325
0,Audio,3169627.74,1827851.77,6625
6,"Music, Movies and Audio Books",3131006.44,1909259.17,7784
4,Games and Toys,724829.43,396668.77,6128


## 8. Analyze sales by subcategory

In [10]:
subcategory_performance = (
    sales.groupby(['Category', 'Subcategory'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
    .sort_values('Revenue', ascending=False)
)

subcategory_performance[['Revenue', 'Profit']] = subcategory_performance[['Revenue', 'Profit']].round(2)
display(subcategory_performance)

,Category,Subcategory,Revenue,Profit,Orders
12,Computers,Desktops,9906356.50,5629155.33,5751
30,TV and Video,Televisions,4308719.19,2631908.38,1723
16,Computers,Projectors & Screens,3767522.00,2357342.37,1468
26,Home Appliances,Water Heaters,3547822.50,2039982.30,1430
3,Cameras and camcorders,Camcorders,3357990.00,2018119.70,1363
13,Computers,Laptops,3164777.20,1793677.71,1543
27,"Music, Movies and Audio Books",Movie DVD,3131006.44,1909259.17,7784
10,Cell phones,Touch Screen Phones,3083462.00,1738593.81,3173
9,Cell phones,Smart phones & PDAs,2805657.00,1590479.02,3130
24,Home Appliances,Refrigerators,2152664.41,1316366.73,704


## 9. Calculate product performance

In [11]:
product_performance = (
    sales.groupby(['ProductKey', 'Product Name'], as_index=False)
    .agg(
        Revenue=('Sales USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Quantity=('Quantity', 'sum'),
        Orders=('Order Number', 'nunique'),
    )
)

product_performance[['Revenue', 'Profit']] = product_performance[['Revenue', 'Profit']].round(2)

## 10. Find the top 10 products by sales

In [12]:
top_products_by_sales = product_performance.nlargest(10, 'Revenue')
display(top_products_by_sales)

,ProductKey,Product Name,Revenue,Profit,Quantity,Orders
442,444,WWI Desktop PC2.33 X2330 Black,505450.00,337986.00,550,154
414,416,Adventure Works Desktop PC2.33 XD233 Silver,466089.00,311663.95,481,146
426,428,Adventure Works Desktop PC2.33 XD233 Brown,464151.00,310368.05,479,144
420,422,Adventure Works Desktop PC2.33 XD233 Black,447678.00,299352.90,462,151
431,433,Adventure Works Desktop PC2.33 XD233 White,437019.00,292225.45,451,141
453,455,WWI Desktop PC2.33 X2330 White,424578.00,283908.24,462,131
448,450,WWI Desktop PC2.33 X2330 Brown,422740.00,282679.20,460,151
146,147,"Adventure Works 52"" LCD HDTV X590 White",394398.64,263727.12,136,40
145,146,"Adventure Works 52"" LCD HDTV X590 Black",374098.71,250152.93,129,44
436,438,WWI Desktop PC2.33 X2330 Silver,360248.00,240891.84,392,141


## 11. Find the top 10 products by profit

In [13]:
top_products_by_profit = product_performance.nlargest(10, 'Profit')
display(top_products_by_profit)

,ProductKey,Product Name,Revenue,Profit,Quantity,Orders
442,444,WWI Desktop PC2.33 X2330 Black,505450.00,337986.00,550,154
414,416,Adventure Works Desktop PC2.33 XD233 Silver,466089.00,311663.95,481,146
426,428,Adventure Works Desktop PC2.33 XD233 Brown,464151.00,310368.05,479,144
420,422,Adventure Works Desktop PC2.33 XD233 Black,447678.00,299352.90,462,151
431,433,Adventure Works Desktop PC2.33 XD233 White,437019.00,292225.45,451,141
453,455,WWI Desktop PC2.33 X2330 White,424578.00,283908.24,462,131
448,450,WWI Desktop PC2.33 X2330 Brown,422740.00,282679.20,460,151
146,147,"Adventure Works 52"" LCD HDTV X590 White",394398.64,263727.12,136,40
145,146,"Adventure Works 52"" LCD HDTV X590 Black",374098.71,250152.93,129,44
436,438,WWI Desktop PC2.33 X2330 Silver,360248.00,240891.84,392,141


## 12. Find the bottom 10 products by sales

In [14]:
bottom_products_by_sales = product_performance.nsmallest(10, 'Revenue')
display(bottom_products_by_sales)

,ProductKey,Product Name,Revenue,Profit,Quantity,Orders
919,921,SV USB Data Cable E600 Silver,15.20,7.52,16,5
924,926,SV USB Sync Charge Cable E700 Silver,15.92,7.84,8,4
2419,2445,Litware 80mm Dual Ball Bearing Case Fan E1001 ...,19.96,9.80,4,2
920,922,SV USB Data Cable E600 Grey,21.85,10.81,23,6
917,919,SV USB Data Cable E600 Pink,25.65,12.69,27,9
2405,2431,"Litware 16"" White Oscillating Stand Fan E701 W...",29.09,14.26,1,1
2413,2439,Litware 120mm Blue LED Case Fan E901 Black,29.97,14.70,3,2
918,920,SV USB Data Cable E600 Black,30.40,15.04,32,10
921,923,SV USB Sync Charge Cable E700 Blue,35.82,17.64,18,9
922,924,SV USB Sync Charge Cable E700 Black,37.81,18.62,19,6


## 13. Find products that generate losses

In [15]:
loss_making_products = product_performance[product_performance['Profit'] < 0]

print('Number of loss-making products:', len(loss_making_products))
display(loss_making_products.sort_values('Profit'))

Number of loss-making products: 0


,ProductKey,Product Name,Revenue,Profit,Quantity,Orders


## 14. Display the Phase 3 answers

In [16]:
print('PHASE 3 ANSWERS')
print('-' * 50)
print(f'Total company revenue: ${total_revenue:,.2f}')
print(f'Total company profit: ${total_profit:,.2f}')
print(f'Average order value: ${average_order_value:,.2f}')
print(f'Profit margin: {profit_margin:.2f}%')
print(f'Month with highest sales: {highest_month["Month Name"]}')
print(f'Highest average sales month: {highest_average_month["Month Name"]}')
print(f'Lowest average sales month: {lowest_average_month["Month Name"]}')
print(f'Year with maximum growth: {int(highest_growth_row["Year"])}')
print(f'Loss-making products: {len(loss_making_products)}')

PHASE 3 ANSWERS
--------------------------------------------------
Total company revenue: $55,755,479.59
Total company profit: $32,662,688.38
Average order value: $2,117.89
Profit margin: 58.58%
Month with highest sales: February
Highest average sales month: December
Lowest average sales month: April
Year with maximum growth: 2018
Loss-making products: 0
